# Model Comparison

This notebook shows how to compare fitted models using the methods that
`statsmodels` provides for the linear regression family (`OLS`, `WLS` and
`GLS`).

For any two nested models, `statsmodels` gives you information criteria
(`AIC`, `BIC`), likelihood-based tests, and `F` tests to decide whether a
more flexible model is worth the extra parameters. We will fit the three
members of the linear regression family to the same dataset and compare them
side by side.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import statsmodels.api as sm

## The data

We use the classic Longley dataset, which is bundled with `statsmodels`.
It is a small macroeconomic time series with six predictors and total
employment as the response.

In [ ]:
data = sm.datasets.longley.load()
data.exog = sm.add_constant(data.exog)
print(data.exog.head())
print(data.exog.columns)

## Baseline: Ordinary Least Squares (OLS)

The OLS model assumes the errors are independent and identically
distributed. It is the natural baseline that the other two estimators
generalize.

In [ ]:
ols_res = sm.OLS(data.endog, data.exog).fit()
print(ols_res.summary())

## Weighted Least Squares (WLS)

If the error variance is not constant but proportional to a known variable,
weighted least squares is more efficient than OLS. Here we weight each
observation by the population (`POP`), i.e. we trust observations from years
with a larger population relatively more.

In [ ]:
weights = data.exog["POP"].to_numpy(dtype=float)
wls_res = sm.WLS(data.endog, data.exog, weights=weights).fit()
print(wls_res.summary())

## Generalized Least Squares (GLS)

When the errors are correlated (as is often the case in time series), GLS
can take that correlation structure into account. We fit the same model but
assume the errors follow an AR(1) process. A consistent estimate of the
autocorrelation $\rho$ is obtained from the OLS residuals, and the
corresponding covariance matrix is used in the GLS fit.

In [ ]:
ols_resid = np.asarray(ols_res.resid)
rho = np.corrcoef(ols_resid[1:], ols_resid[:-1])[0, 1]
n = len(data.endog)
sigma = rho ** np.abs(np.subtract.outer(np.arange(n), np.arange(n)))
gls_res = sm.GLS(data.endog, data.exog, sigma=sigma).fit()
print(gls_res.summary())

## Comparing the models

The simplest comparison uses the information criteria. Lower values of `AIC`
and `BIC` indicate a better fit after penalizing model complexity. The
log-likelihood (`llf`) measures how well the model fits the data, and
`R-squared` is the usual coefficient of determination.

In [ ]:
results = [ols_res, wls_res, gls_res]
names = ["OLS", "WLS", "GLS"]

print(f"{'Model':<8}{'AIC':>12}{'BIC':>12}{'log-L':>12}{'R2':>10}")
for name, res in zip(names, results):
    print(
        f"{name:<8}"
        f"{res.aic:>12.3f}"
        f"{res.bic:>12.3f}"
        f"{res.llf:>12.3f}"
        f"{res.rsquared:>10.4f}"
    )

## Comparing nested models

`statsmodels` also provides formal hypothesis tests for nested models. The
methods `compare_lr_test` (likelihood ratio test) and `compare_f_test`
(Chow-style F test) compare a larger model against a restricted model that
drops one or more regressors.

As an example, we take the baseline OLS model and remove the `ARMED`
regressor. Both tests tell us whether dropping that regressor significantly
worsens the fit.

In [ ]:
exog_restricted = data.exog.drop(columns=["ARMED"])
restricted_res = sm.OLS(data.endog, exog_restricted).fit()

lr_stat, lr_pvalue, lr_df = ols_res.compare_lr_test(restricted_res)
f_stat, f_pvalue, f_df = ols_res.compare_f_test(restricted_res)

print(f"Likelihood ratio test: stat={lr_stat:.3f}, p-value={lr_pvalue:.4f}, df={lr_df}")
print(f"F test:                stat={f_stat:.3f}, p-value={f_pvalue:.4f}, df={f_df}")

## Visual check of residuals

Finally, a quick visual comparison of the residuals against fitted values
for the three estimators. Ideally the residuals scatter randomly around
zero with no obvious pattern.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, name, res in zip(axes, names, results):
    ax.scatter(res.fittedvalues, res.resid, alpha=0.6)
    ax.axhline(0, color="k", linestyle="--", linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")
plt.tight_layout()
plt.show()

## Summary

In this example the GLS estimator, which accounts for the serial
correlation in the residuals, achieves the lowest `AIC` and `BIC` among the
three models. The nested-model tests show that `ARMED` is a statistically
significant predictor, so it should be kept in the model.

The key takeaway: for nested models `compare_lr_test` and `compare_f_test`
give you a formal answer, and for non-nested models the information criteria
(`AIC`/`BIC`) are the tool to use.